In [ ]:
from pathlib import Path

from matplotlib import pyplot as plt
from sbi.analysis import pairplot

from mach3sbitools.inference import InferenceHandler
from mach3sbitools.simulator import Simulator
from mach3sbitools.utils import MaCh3Logger, PosteriorConfig, TrainingConfig

MaCh3Logger("mach3sbitools")

In [ ]:
# Some global variables
BASE = Path("jupyter_tutorial")

DATA_FILE = BASE / "data/my_data.feather"
PRIOR_FILE = BASE / "prior/my_prior.pkl"
MODEL_FILE = BASE / "model/my_model.ckpt"
INFERENCE_FILE = BASE / "inference/my_inference.paraquet"

In [ ]:
# Firstly we load the simulator
simulator = Simulator(
    module_name="my_simulator",
    class_name="MySimulator",
    config=Path("physics_configs/PhysicsConfig.yaml"),
)

theta, x = simulator.simulate(1000000)
simulator.save(DATA_FILE, theta, x)

simulator.prior.save(PRIOR_FILE)


In [ ]:
import yaml
from physics_engine import ParameterHandler, SampleHandler
from tqdm.autonotebook import tqdm

physics_config=Path("physics_configs/PhysicsConfig.yaml")
# First we get the config and check it exists

if not physics_config.is_file() and not physics_config.exists():
    raise FileNotFoundError("Config file not found.")

# We extract the sample config from the physics config. This is a tad hacky but
# is what's currently done in MaCh3
with open(physics_config) as f:
    physics_open = yaml.safe_load(f)
    sample_config = Path(physics_open["Sample"]["sample_config"])

# We also need to get the SAMPLE config from the physics config!
if not sample_config.is_file():
    raise ValueError("Config file not found.")

parameter_handler = ParameterHandler(physics_config)
sample_handler = SampleHandler(sample_config, parameter_handler)

simulator_llhs = [simulator.simulator_wrapper.get_llh(t) for t in tqdm(theta)]

In [ ]:
# Now we setup training
posterior_config = PosteriorConfig(
    model="maf",
    hidden_features=64,
    num_transforms=5,
    dropout_probability=0.2,
)

training_config = TrainingConfig(
    save_path=MODEL_FILE,
    max_epochs=2000,
    learning_rate=1e-5,
    show_progress=True,
    tensorboard_dir=BASE/"tensor-logs",
)

trainer = InferenceHandler(PRIOR_FILE)
trainer.set_dataset(DATA_FILE.parent)
trainer.load_training_data()
trainer.create_posterior(posterior_config)

In [ ]:
# Now we run training
trainer.train_posterior(training_config, model_config=posterior_config)

In [ ]:
# data_bins = simulator.simulator_wrapper.get_data_bins()
data_bins = simulator.simulator_wrapper.get_data_bins()
print(data_bins)
samples = trainer.sample_posterior(1_000_000, data_bins)

In [ ]:
pairplot(samples, labels=simulator.simulator_wrapper.get_parameter_names())
plt.show()